<a href="https://colab.research.google.com/github/Dharmendra0305/PRODIGY_GA_04/blob/main/Image_to_Image_Translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Phase 1: Choose the Dataset  and Set Up the Environment**

In [ ]:
import tensorflow as tf
import os

In [ ]:
# 1. Download and extract the dataset
url = "http://efrosgans.eecs.berkeley.edu/pix2pix/datasets/maps.tar.gz"
path_to_zip = tf.keras.utils.get_file('maps.tar.gz', origin = url, extract = True)
PATH = os.path.join(os.path.dirname(path_to_zip), 'maps_extracted', 'maps')

# 2. Function to load, split, and normalize the images
def load_image(image_path):
  # Read and decode the image
  img = tf.io.read_file(image_path)
  img = tf.image.decode_jpeg(img)

  # The images are stitched together side-by-side
  # We split them exactly down the middle
  w = tf.shape(img)[1] // 2
  satellite_img = img[:, :w, :]
  map_img = img[:, w:, :]

  # Resize both halves to 256x256
  satellite_img = tf.image.resize(satellite_img, [256, 256])
  map_img = tf.image.resize(map_img, [256, 256])

  # Normalize pixel values to the range [-1, 1] (crucial for GANs)
  satellite_img = tf.cast(satellite_img, tf.float32) / 127.5 - 1.0
  map_img = tf.cast(map_img, tf.float32) / 127.5 - 1.0

  return satellite_img, map_img

# Test it on one training image
sample_file = os.path.join(PATH, "train/910.jpg")
sat, map_target = load_image(sample_file)
print(f"Iamge Shapes: Satellite {sat.shape}, Map {map_target.shape}")

# **Phase 2: Build the Generator (U-Net)**

In [ ]:
from tensorflow.keras import layers

def downsample(filters, size, apply_batchnorm=True):
    """The Encoder part: Shrinks the image and extracts features."""
    initializer = tf.random_normal_initializer(0., 0.02)
    result = tf.keras.Sequential()
    result.add(layers.Conv2D(filters, size, strides=2, padding='same',
                             kernel_initializer=initializer, use_bias=False))
    if apply_batchnorm:
        result.add(layers.BatchNormalization())
    result.add(layers.LeakyReLU())
    return result

def upsample(filters, size, apply_dropout=False):
    """The Decoder part: Expands the image back up."""
    initializer = tf.random_normal_initializer(0., 0.02)
    result = tf.keras.Sequential()
    result.add(layers.Conv2DTranspose(filters, size, strides=2, padding='same',
                                      kernel_initializer=initializer, use_bias=False))
    result.add(layers.BatchNormalization())
    if apply_dropout:
        result.add(layers.Dropout(0.5))
    result.add(layers.ReLU())
    return result

def build_generator():
    """Assembles the U-Net with skip connections."""
    inputs = layers.Input(shape=[256, 256, 3])

    # Encoder (Downsampling)
    down_stack = [
        downsample(64, 4, apply_batchnorm=False), # (128, 128, 64)
        downsample(128, 4),                       # (64, 64, 128)
        downsample(256, 4),                       # (32, 32, 256)
        downsample(512, 4),                       # (16, 16, 512)
        downsample(512, 4),                       # (8, 8, 512)
        downsample(512, 4),                       # (4, 4, 512)
        downsample(512, 4),                       # (2, 2, 512)
        downsample(512, 4),                       # (1, 1, 512)
    ]

    # Decoder (Upsampling)
    up_stack = [
        upsample(512, 4, apply_dropout=True),     # (2, 2, 1024)
        upsample(512, 4, apply_dropout=True),     # (4, 4, 1024)
        upsample(512, 4, apply_dropout=True),     # (8, 8, 1024)
        upsample(512, 4),                         # (16, 16, 1024)
        upsample(256, 4),                         # (32, 32, 512)
        upsample(128, 4),                         # (64, 64, 256)
        upsample(64, 4),                          # (128, 128, 128)
    ]

    initializer = tf.random_normal_initializer(0., 0.02)
    last = layers.Conv2DTranspose(3, 4, strides=2, padding='same',
                                  kernel_initializer=initializer, activation='tanh') # (256, 256, 3)

    x = inputs
    skips = []
    # Route through encoder
    for down in down_stack:
        x = down(x)
        skips.append(x)

    skips = reversed(skips[:-1])

    # Route through decoder and add skip connections
    for up, skip in zip(up_stack, skips):
        x = up(x)
        x = layers.Concatenate()([x, skip])

    x = last(x)
    return tf.keras.Model(inputs=inputs, outputs=x)

# Create the generator model
generator = build_generator()
print("Generator built successfully!")

# **Phase 3: Build the Discriminator (PatchGAN)**

In [ ]:
from tensorflow.keras import layers

def build_discriminator():
    """Builds a PatchGAN Discriminator."""
    initializer = tf.random_normal_initializer(0., 0.02)

    # Inputs: The satellite image and the map image (which could be real or fake)
    inp = layers.Input(shape=[256, 256, 3], name='input_image')
    tar = layers.Input(shape=[256, 256, 3], name='target_image')

    # Concatenate them so the discriminator sees them together
    x = layers.concatenate([inp, tar]) # Shape: (256, 256, 6)

    # Layer 1
    x = layers.Conv2D(64, 4, strides=2, padding='same',
                      kernel_initializer=initializer)(x)
    x = layers.LeakyReLU()(x)

    # Layer 2
    x = layers.Conv2D(128, 4, strides=2, padding='same',
                      kernel_initializer=initializer, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    # Layer 3
    x = layers.Conv2D(256, 4, strides=2, padding='same',
                      kernel_initializer=initializer, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    # Layer 4 (Padding and Convolution)
    x = layers.ZeroPadding2D()(x)
    x = layers.Conv2D(512, 4, strides=1,
                      kernel_initializer=initializer, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    # Output Layer (Outputs a 30x30 grid of values instead of 1 value)
    x = layers.ZeroPadding2D()(x)
    last = layers.Conv2D(1, 4, strides=1,
                         kernel_initializer=initializer)(x)

    return tf.keras.Model(inputs=[inp, tar], outputs=last)

discriminator = build_discriminator()
print("Discriminator built successfully!")

# **Phase 4: Define Losses and Train**

In [ ]:
# Standard loss function for binary classification (Real=1, Fake=0)
loss_object = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(disc_real_output, disc_generated_output):
    """Calculates how well the Discriminator spots real vs. fake."""
    real_loss = loss_object(tf.ones_like(disc_real_output), disc_real_output)
    generated_loss = loss_object(tf.zeros_like(disc_generated_output), disc_generated_output)
    return real_loss + generated_loss

LAMBDA = 100 # Weight for the L1 loss

def generator_loss(disc_generated_output, gen_output, target):
    """Calculates how well the Generator fools the Discriminator AND matches the target map."""
    # Adversarial loss (Did it fool the discriminator?)
    gan_loss = loss_object(tf.ones_like(disc_generated_output), disc_generated_output)

    # L1 loss (Pixel-by-pixel difference from the real map)
    l1_loss = tf.reduce_mean(tf.abs(target - gen_output))

    return gan_loss + (LAMBDA * l1_loss)

# Optimizers to update the network weights
generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

@tf.function
def train_step(input_image, target):
    """One single step of training."""
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        # 1. Generate a fake map
        gen_output = generator(input_image, training=True)

        # 2. Evaluate real and fake maps
        disc_real_output = discriminator([input_image, target], training=True)
        disc_generated_output = discriminator([input_image, gen_output], training=True)

        # 3. Calculate the losses
        gen_total_loss = generator_loss(disc_generated_output, gen_output, target)
        disc_loss = discriminator_loss(disc_real_output, disc_generated_output)

    # 4. Calculate gradients (the direction to adjust the weights)
    generator_gradients = gen_tape.gradient(gen_total_loss, generator.trainable_variables)
    discriminator_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    # 5. Apply the updates
    generator_optimizer.apply_gradients(zip(generator_gradients, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(discriminator_gradients, discriminator.trainable_variables))

# **Phase 5: Final Training and Visualization**

In [ ]:
import matplotlib.pyplot as plt
import time
from IPython import display

def generate_images(model, test_input, tar):
    """Generates the side-by-side graphic for your portfolio."""
    # Generate the fake image
    prediction = model(test_input, training=True)
    plt.figure(figsize=(15, 5))

    display_list = [test_input[0], tar[0], prediction[0]]
    title = ['Input (Satellite)', 'Target (Real Map)', 'AI Generated Map']

    for i in range(3):
        plt.subplot(1, 3, i+1)
        plt.title(title[i])
        # Rescale pixels from [-1, 1] back to [0, 1] for displaying
        plt.imshow(display_list[i] * 0.5 + 0.5)
        plt.axis('off')
    plt.show()

def fit(epochs):
    """The main training loop."""
    # Convert our single sample image into a "batch" of 1
    input_batch = tf.expand_dims(sat, 0)
    target_batch = tf.expand_dims(map_target, 0)

    for epoch in range(epochs):
        start = time.time()

        # 1. Train the models on the image
        train_step(input_batch, target_batch)

        # 2. Show progress every 10 epochs
        if (epoch + 1) % 10 == 0:
            display.clear_output(wait=True)
            print(f"Epoch {epoch+1}/{epochs}")
            generate_images(generator, input_batch, target_batch)
            print(f'Time for epoch {epoch + 1} is {time.time()-start:.2f} sec\n')

# Start training! (100 epochs is enough to see it start working)
fit(100)

# **Phase 6: Interactive Web App**

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import numpy as np

def generate_map(input_image):
    # 1. Resize and normalize the user's uploaded image to match training data
    img = tf.image.resize(input_image, [256, 256])
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    img = tf.expand_dims(img, 0) # Add the batch dimension

    # 2. Generate the map using your trained model
    prediction = generator(img, training=True)

    # 3. Process the output back into a standard viewable image
    output_img = prediction[0].numpy()
    output_img = (output_img * 0.5 + 0.5) * 255 # Rescale back to 0-255

    return output_img.astype(np.uint8)

# Build the layout
app = gr.Interface(
    fn=generate_map,
    inputs=gr.Image(label="Upload Satellite Image"),
    outputs=gr.Image(label="AI Generated Map"),
    title="Pix2Pix: Satellite to Map",
    description="My Internship Task 04: Upload a satellite image and see the cGAN generate a map layout!"
)

# Launch it!
app.launch(share=True)